In [8]:
library(tidyverse)
library(oce)
source('../DATA/interpolateData.R')

# =============================================================================
# STEADY-STATE FORCING CALCULATION
# Following Stock et al. (2008) chemostat framework
#
# Depth intervals are defined here and can be adjusted per regime.
# =============================================================================

niskin_ds_3 <- readRDS("../DATA/processed/Niskin_cleaned.rds")
CARIACO     <- readRDS("../DATA/processed/CARIACO_EnvData_combined.rds")

# ---- Define depth intervals (adjust per regime as needed) ----
euphotic_lower <- 50
N0_upper       <- 50
N0_lower       <- 70

# ---- Extract depth-specific values using interpolateData ----

# NO3 euphotic zone mean (0-50m) — model state variable N
NO3_euphotic_ds <- interpolateData(
  niskin_ds_3, "NO3_merged",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 10,
  output_type = 'mean', surface_fix = TRUE
)
names(NO3_euphotic_ds)[1] <- "NO3_euphotic"

write.csv(NO3_euphotic_ds, "NO3_euphotic_dynamic.csv", row.names = FALSE)
cat("Saved to NO3_euphotic_dynamic.csv\n")

# NO3 sub-euphotic mean (50-70m) — model forcing N0
NO3_subeuphotic_ds <- interpolateData(
  niskin_ds_3, "NO3_merged",
  depth_from = N0_upper, depth_to = N0_lower, noofNA = 10,
  output_type = 'mean', surface_fix = FALSE
)
names(NO3_subeuphotic_ds)[1] <- "NO3_subeuphotic"

# Temperature euphotic zone mean (0-50m) — for f-ratio
Temp_euphotic_ds <- interpolateData(
  niskin_ds_3, "Temperature",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 10,
  output_type = 'mean', surface_fix = TRUE
)
names(Temp_euphotic_ds)[1] <- "Temp_euphotic"

# Primary Productivity integrated (0-50m)
PP_ds <- interpolateData(
  niskin_ds_3, "PrimaryProductivity",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 20,
  output_type = 'integrated', surface_fix = TRUE
)
names(PP_ds)[1] <- "PP_integrated"

PP_ds$PP_integrated <- PP_ds$PP_integrated * 12  # hr -> day (12h tropical daylight)

# PON (euphotic zone)
PON_obs <- interpolateData(
  niskin_ds_3, "PN_ug_L",
  depth_from = 0, depth_to = euphotic_lower,
  noofNA = 10, output_type = 'mean', surface_fix = TRUE
)
PON_obs$PON_euphotic <- PON_obs$dat_var / 14.007  # µg/L -> mmol N m-3
PON_obs$dat_var <- NULL

write.csv(PON_obs, "PON_euphotic_dynamic.csv", row.names = FALSE)
cat("Saved to PON_euphotic_dynamic.csv\n")

# =============================================================================
# SEDIMENT TRAP EXPORT FLUX
# Back-calculate export at model boundary from 225m trap
# =============================================================================

# ---- Extract 225m trap data from combined dataset ----
mean_trap_225 <- mean(CARIACO$MF_N_mmol_225m, na.rm = TRUE)
sd_trap_225   <- sd(CARIACO$MF_N_mmol_225m, na.rm = TRUE)
n_trap_225    <- sum(!is.na(CARIACO$MF_N_mmol_225m))

cat(sprintf("\n=== Sediment Trap (225m) ===\n"))
cat(sprintf("  Mean flux:    %.4f mmol N m-2 d-1\n", mean_trap_225))
cat(sprintf("  SD:           %.4f mmol N m-2 d-1\n", sd_trap_225))
cat(sprintf("  n months:     %d\n", n_trap_225))

# ---- Martin curve back-calculation to model boundary ----
z_model   <- euphotic_lower  # 50m
z_trap    <- 225
martin_b  <- 0.858           # canonical value

correction_factor <- (z_trap / z_model)^martin_b

export_50m_mean <- mean_trap_225 * correction_factor
export_50m_sd   <- sd_trap_225 * correction_factor
export_volumetric <- export_50m_mean / euphotic_lower  # mmol N m-3 d-1

cat(sprintf("\n=== Export Flux at Model Boundary (%dm) ===\n", z_model))
cat(sprintf("  Martin b (canonical):  %.3f\n", martin_b))
cat(sprintf("  Correction factor:     %.2f\n", correction_factor))
cat(sprintf("  Export flux (mean):    %.4f mmol N m-2 d-1\n", export_50m_mean))
cat(sprintf("  Export flux (SD):      %.4f mmol N m-2 d-1\n", export_50m_sd))
cat(sprintf("  Volumetric equivalent: %.6f mmol N m-3 d-1\n", export_volumetric))

# =============================================================================
# DATA AVAILABILITY
# =============================================================================

cat("\n=== Data availability ===\n")
cat(sprintf("  NO3 euphotic (0-%dm):      %d dates\n", euphotic_lower, nrow(NO3_euphotic_ds)))
cat(sprintf("  NO3 sub-euphotic (%d-%dm): %d dates\n", N0_upper, N0_lower, nrow(NO3_subeuphotic_ds)))
cat(sprintf("  Temperature (0-%dm):       %d dates\n", euphotic_lower, nrow(Temp_euphotic_ds)))
cat(sprintf("  PP integrated (0-%dm):     %d dates\n", euphotic_lower, nrow(PP_ds)))
cat(sprintf("  PON euphotic (0-%dm):      %d dates\n", euphotic_lower, nrow(PON_obs)))
cat(sprintf("  Sediment trap (225m):      %d months\n", n_trap_225))

# =============================================================================
# GRAND MEANS
# =============================================================================

mean_euphotic_depth <- mean(CARIACO$euphotic_depth, na.rm = TRUE)
mean_N0        <- mean(NO3_subeuphotic_ds$NO3_subeuphotic, na.rm = TRUE)
mean_N_surface <- mean(NO3_euphotic_ds$NO3_euphotic, na.rm = TRUE)
mean_temp      <- mean(Temp_euphotic_ds$Temp_euphotic, na.rm = TRUE)
mean_PP        <- mean(PP_ds$PP_integrated, na.rm = TRUE)
mean_PON       <- mean(PON_obs$PON_euphotic, na.rm = TRUE)
mean_export    <- export_50m_mean

cat("\n=== Grand Mean Values ===\n")
cat(sprintf("  Euphotic depth:       %.2f m\n", mean_euphotic_depth))
cat(sprintf("  Temperature (0-%dm):  %.2f °C\n", euphotic_lower, mean_temp))
cat(sprintf("  PP (integrated):      %.2f mg C m-2 d-1\n", mean_PP))
cat(sprintf("  N0 (%d-%dm NO3):      %.4f mmol N m-3\n", N0_upper, N0_lower, mean_N0))
cat(sprintf("  N_surface (0-%dm):    %.4f mmol N m-3\n", euphotic_lower, mean_N_surface))
cat(sprintf("  PON (0-%dm):          %.4f mmol N m-3\n", euphotic_lower, mean_PON))
cat(sprintf("  Export flux (%dm):    %.4f mmol N m-2 d-1 (from 225m trap)\n", z_model, mean_export))

# =============================================================================
# DERIVED FORCING
# =============================================================================

# ---- f-ratio (Laws et al. 2011) ----
f_ratio_calc <- function(T, PP_mgC) {
  (0.5857 - 0.0165 * T) * PP_mgC / (51.7 + PP_mgC)
}

f_ratio <- f_ratio_calc(mean_temp, mean_PP)

# ---- New production ----
new_prod_area     <- f_ratio * mean_PP                                        # mg C m-2 d-1
new_nutrient_flux <- new_prod_area / mean_euphotic_depth / 12.01 * (16 / 106) # mmol N m-3 d-1

# ---- Dilution rate ----
delta_N       <- mean_N0 - mean_N_surface
dilution_rate <- new_nutrient_flux / delta_N

cat(sprintf("\n=== Derived Forcing ===\n"))
cat(sprintf("  f-ratio:              %.4f\n", f_ratio))
cat(sprintf("  New production:       %.4f mg C m-2 d-1\n", new_prod_area))
cat(sprintf("  Nutrient flux:        %.6f mmol N m-3 d-1\n", new_nutrient_flux))
cat(sprintf("  N0 - N_surface:       %.4f mmol N m-3\n", delta_N))
cat(sprintf("  Dilution rate (d):    %.6f d-1\n", dilution_rate))
cat(sprintf("  Residence time (1/d): %.1f days\n", 1 / dilution_rate))

# =============================================================================
# CONSISTENCY CHECK: Export vs New Production
# =============================================================================

new_prod_N <- new_prod_area / 12.01 * (16/106)  # mg C m-2 d-1 -> mmol N m-2 d-1

cat(sprintf("\n=== Consistency Check ===\n"))
cat(sprintf("  New production:    %.4f mmol N m-2 d-1 (f-ratio method)\n", new_prod_N))
cat(sprintf("  Export flux:       %.4f mmol N m-2 d-1 (trap-derived)\n", mean_export))
cat(sprintf("  Ratio (exp/new):   %.2f\n", mean_export / new_prod_N))

# =============================================================================
# SUMMARY TABLE
# =============================================================================

forcing_summary <- tibble(
  parameter = c("euphotic_depth", "temperature", "PP_total",
                "f_ratio", "new_production_area", "new_nutrient_flux",
                "N0", "N_surface", "N0_minus_N", "dilution_rate", "residence_time",
                "PON", "export_flux", "martin_b"),
  value = c(mean_euphotic_depth, mean_temp, mean_PP,
            f_ratio, new_prod_area, new_nutrient_flux,
            mean_N0, mean_N_surface, delta_N, dilution_rate, 1 / dilution_rate,
            mean_PON, mean_export, martin_b),
  units = c("m", "deg_C", "mg_C_m-2_d-1",
            "dimensionless", "mg_C_m-2_d-1", "mmol_N_m-3_d-1",
            "mmol_N_m-3", "mmol_N_m-3", "mmol_N_m-3", "d-1", "days",
            "mmol_N_m-3", "mmol_N_m-2_d-1", "dimensionless"),
  depth_interval = c(
    "from_combined_dataset",
    sprintf("0-%dm", euphotic_lower),
    sprintf("0-%dm_integrated", euphotic_lower),
    "derived_Laws2011", "derived", "derived",
    sprintf("%d-%dm", N0_upper, N0_lower),
    sprintf("0-%dm", euphotic_lower),
    "derived", "derived", "derived",
    sprintf("0-%dm", euphotic_lower),
    sprintf("%dm_from_225m_trap", z_model),
    "Martin_1987")
)

print(forcing_summary)
write.csv(forcing_summary, "steady_state_forcing_summary.csv", row.names = FALSE)
cat("\nSaved to steady_state_forcing_summary.csv\n")

Saved to NO3_euphotic_dynamic.csv
Saved to PON_euphotic_dynamic.csv

=== Sediment Trap (225m) ===
  Mean flux:    0.7104 mmol N m-2 d-1
  SD:           0.4274 mmol N m-2 d-1
  n months:     139

=== Export Flux at Model Boundary (50m) ===
  Martin b (canonical):  0.858
  Correction factor:     3.63
  Export flux (mean):    2.5821 mmol N m-2 d-1
  Export flux (SD):      1.5534 mmol N m-2 d-1
  Volumetric equivalent: 0.051641 mmol N m-3 d-1

=== Data availability ===
  NO3 euphotic (0-50m):      211 dates
  NO3 sub-euphotic (50-70m): 219 dates
  Temperature (0-50m):       226 dates
  PP integrated (0-50m):     219 dates
  PON euphotic (0-50m):      214 dates
  Sediment trap (225m):      139 months

=== Grand Mean Values ===
  Euphotic depth:       44.92 m
  Temperature (0-50m):  24.34 °C
  PP (integrated):      1202.97 mg C m-2 d-1
  N0 (50-70m NO3):      5.5564 mmol N m-3
  N_surface (0-50m):    2.0158 mmol N m-3
  PON (0-50m):          9.4094 mmol N m-3
  Export flux (50m):    2.5821 m